In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(18)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 18)
library(SeuratDisk)
library(reshape2)
library(tidyverse)
library(RColorBrewer)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/"
repDir <- paste0(mainDir, "composition_DEG_signatures/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- paste0(mainDir, "refs/")


dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

In [ ]:
### Load object
seu <- readRDS(paste0(mainDir, "/QC_clustering/merged_clusters.rds"))

In [ ]:
### Differential gene expression analysis

# Perform differential expression analysis for genotype
de_results <- FindMarkers(
  object = seu,
  ident.1 = "gal",
  ident.2 = "flp",
  group.by = "genotype",
  min.pct = 0.25,
  logfc.threshold = 0.25
)

filtered_results <- subset(de_results, p_val < 0.05 & abs(avg_log2FC) > 0.5)
write.csv(filtered_results, file.path(tabDir, "DE_results_gal_vs_flp.csv"))

top_genes <- head(rownames(filtered_results[order(-abs(filtered_results$avg_log2FC)), ]), 10)

### Combined violin across genotype
expr_data <- FetchData(seu, vars = c(top_genes, "genotype"))
expr_data_long <- pivot_longer(expr_data, cols = all_of(top_genes), names_to = "gene", values_to = "expression")

expr_data_long$genotype <- factor(expr_data_long$genotype, levels = c("gal", "flp"))

vln_genotype <- ggplot(expr_data_long, aes(x = gene, y = expression, fill = genotype)) +
  geom_violin(position = position_dodge(0.8), width = 0.7, scale = "width", trim = TRUE) +
  stat_summary(fun = median, geom = "point", position = position_dodge(0.8), size = 0.8, color = "black") +
  theme_minimal() +
  labs(title = "Top 10 DE Genes: gal vs flp", x = "Gene", y = "Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14),
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.title = element_blank()
  )



# Perform differential expression analysis for timepoint
de_results <- FindMarkers(
  object = seu,
  ident.1 = "10d",
  ident.2 = "12d",
  group.by = "timepoint",
  min.pct = 0.25,
  logfc.threshold = 0.25
)

filtered_results <- subset(de_results, p_val < 0.05 & abs(avg_log2FC) > 0.5)
write.csv(filtered_results, file.path(tabDir, "DE_results_10d_vs_12d.csv"))

top_genes <- head(rownames(filtered_results[order(-abs(filtered_results$avg_log2FC)), ]), 10)

###### combined timepoint violin plot
expr_data <- FetchData(seu, vars = c(top_genes, "timepoint"))
expr_data_long <- pivot_longer(expr_data, cols = all_of(top_genes), names_to = "gene", values_to = "expression")

expr_data_long$timepoint <- factor(expr_data_long$timepoint, levels = c("10d", "12d"))

vln_timepoint <- ggplot(expr_data_long, aes(x = gene, y = expression, fill = timepoint)) +
  geom_violin(position = position_dodge(0.8), width = 0.7, scale = "width", trim = TRUE) +
  stat_summary(fun = median, geom = "point", position = position_dodge(0.8), size = 0.8, color = "black") +
  theme_minimal() +
  labs(title = "Top 10 DE Genes: 10d vs 12d", x = "Gene", y = "Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14),
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.title = element_blank()
  )

combined_plot <- vln_genotype / vln_timepoint

pdf(file.path(figDir, "Combined_DE_violin_full.pdf"), width = 10, height = 10)
print(combined_plot)
dev.off()

In [ ]:
Idents(seu) <- "seurat_clusters"
cluster_ids <- unique(seu$seurat_clusters)

# Get top DE genes and make combined violin
create_combined_violin <- function(seu_subset, top_genes, group_var, title_prefix) {
  expr_data <- FetchData(seu_subset, vars = c(top_genes, group_var))
  expr_data_long <- pivot_longer(expr_data, cols = all_of(top_genes), names_to = "gene", values_to = "expression")
  expr_data_long[[group_var]] <- factor(expr_data_long[[group_var]])

  p <- ggplot(expr_data_long, aes(x = gene, y = expression, fill = .data[[group_var]])) +
    geom_violin(position = position_dodge(0.8), width = 0.7, scale = "width", trim = TRUE) +
    stat_summary(fun = median, geom = "point", position = position_dodge(0.8), size = 0.6, color = "black") +
    theme_minimal() +
    labs(title = paste0(title_prefix, ": Top DE Genes by ", group_var), x = "Gene", y = "Expression") +
    theme(
      plot.title = element_text(hjust = 0.5, size = 12),
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.title = element_blank()
    )
  return(p)
}

# Loop over clusters
for (cluster_id in cluster_ids) {
  cluster_id_chr <- as.character(cluster_id)
  cluster_obj <- subset(seu, idents = cluster_id)

  # Genotype DE
  de_genotype <- FindMarkers(
    object = cluster_obj,
    ident.1 = "gal",
    ident.2 = "flp",
    group.by = "genotype",
    min.pct = 0.25,
    logfc.threshold = 0.25
  )
  filtered_genotype <- subset(de_genotype, p_val < 0.05 & abs(avg_log2FC) > 0.5)
  write.csv(filtered_genotype, file.path(tabDir, paste0("DE_results_cluster_", cluster_id_chr, "_gal_vs_flp.csv")))
  top_genes_genotype <- head(rownames(filtered_genotype[order(-abs(filtered_genotype$avg_log2FC)), ]), 10)

  # Timepoint DE
  de_timepoint <- FindMarkers(
    object = cluster_obj,
    ident.1 = "10d",
    ident.2 = "12d",
    group.by = "timepoint",
    min.pct = 0.25,
    logfc.threshold = 0.25
  )
  filtered_timepoint <- subset(de_timepoint, p_val < 0.05 & abs(avg_log2FC) > 0.5)
  write.csv(filtered_timepoint, file.path(tabDir, paste0("DE_results_cluster_", cluster_id_chr, "_10d_vs_12d.csv")))
  top_genes_timepoint <- head(rownames(filtered_timepoint[order(-abs(filtered_timepoint$avg_log2FC)), ]), 10)

  vln_genotype <- create_combined_violin(cluster_obj, top_genes_genotype, "genotype", paste("Cluster", cluster_id_chr))
  vln_timepoint <- create_combined_violin(cluster_obj, top_genes_timepoint, "timepoint", paste("Cluster", cluster_id_chr))

  combined_plot <- vln_genotype / vln_timepoint

  pdf(file.path(figDir, paste0("Combined_DE_violin_cluster_", cluster_id_chr, ".pdf")), width = 10, height = 10)
  print(combined_plot)
  dev.off()
}

In [ ]:
### Create count tables
count_table_clusters_samples <- table(seu@meta.data$seurat_clusters, seu@meta.data[["timepoint"]])
write.csv(count_table_clusters_samples, file.path(tabDir, "counts_per_cluster_per_timepoint.csv"))

count_table_clusters_samples <- table(seu@meta.data$seurat_clusters, seu@meta.data[["condition"]])
write.csv(count_table_clusters_samples, file.path(tabDir, "counts_per_cluster_per_condition.csv"))

count_table_clusters_samples <- table(seu@meta.data$seurat_clusters, seu@meta.data[["genotype"]])
write.csv(count_table_clusters_samples, file.path(tabDir, "counts_per_cluster_per_genotype.csv"))

In [ ]:
# Table: condition vs cluster
ta <- melt(t(table(seu$condition, seu$seurat_clusters)))
colnames(ta) <- c("condition", "cluster", "ncells")
ta$condition <- as.factor(ta$condition)
ta$cluster <- as.factor(ta$cluster)

g <- ggplot(aes(condition, ncells, fill=cluster), data=ta) +
  geom_bar(stat="identity") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

g1 <- ggplot(aes(cluster, ncells, fill=cluster), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~condition, ncol=4) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

g2 <- ggplot(aes(condition, ncells, fill=condition), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~cluster, ncol=18) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

pdf(paste0(figDir, "barplot_cells_per_cluster_cond.pdf"), width=10, height=5)
print(g)
print(g1)
print(g2)
dev.off()

write.csv(ta, file=paste0(tabDir, "barplot_cells_per_cluster_cond.csv"))

# Cell cycle by cluster
ta <- melt(t(table(seu$seurat_clusters, seu$Phase)))
colnames(ta) <- c("cluster", "Phase", "ncells")
ta$Phase <- as.factor(ta$Phase)
ta$cluster <- as.factor(ta$cluster)

g <- ggplot(aes(cluster, ncells, fill=Phase), data=ta) +
  geom_bar(stat="identity") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

g1 <- ggplot(aes(Phase, ncells, fill=Phase), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~cluster, ncol=4) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

g2 <- ggplot(aes(cluster, ncells, fill=cluster), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~Phase, ncol=16) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

pdf(paste0(figDir, "barplot_cells_per_Phase_clust.pdf"), width=10, height=5)
print(g)
print(g1)
print(g2)
dev.off()

write.csv(ta, file=paste0(tabDir, "barplot_cells_per_Phase_cond.csv"))

### Double normalisation of cluster composition for library size and cluster size
ta <- melt(table(seu$condition, seu$seurat_clusters))
colnames(ta) <- c("condition", "cluster", "ncells")
ta$condition <- as.factor(ta$condition)
ta$cluster <- as.factor(ta$cluster)
# Normalise to percentage within each condition
ta$ncells_condition_perc <- ave(ta$ncells, ta$condition, FUN=function(x) x / sum(x) * 100)
# Normalise to percentage within each cluster
ta$ncells_cluster_perc <- ave(ta$ncells, ta$cluster, FUN=function(x) x / sum(x) * 100)
# Relative to total cells (double normalization)
# Observed proportion of cells per cluster from each condition / expected proportion of cells per cluster from each condition if equally distributed
# =1 equal distribution to the expected. >1 more cells in the cluster from the condition than expected, <1 less cells in the cluster from the condition than expected
total_cells <- sum(ta$ncells)
ta$composition_normalised_for_library_size <- (ta$ncells / ave(ta$ncells, ta$condition, FUN=sum)) / 
                         (ave(ta$ncells, ta$cluster, FUN=sum) / total_cells)

write.csv(ta, file=paste0(tabDir, "cells_per_cluster_cond_multiple_normalizations.csv"))

g <- ggplot(aes(condition, composition_normalised_for_library_size, fill=condition), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~cluster, ncol=4) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

pdf(paste0(figDir, "cluster_composition_normalised_for_library_size.pdf"), width=12, height=8)
print(g)
dev.off()

### Double normalised phase composition for library size and cluster size
ta <- melt(table(seu$Phase, seu$seurat_clusters))
colnames(ta) <- c("Phase", "cluster", "ncells")
ta$cluster <- as.factor(ta$cluster)
ta$Phase <- as.factor(ta$Phase)
# Normalise to percentage within each condition
ta$ncells_cluster_perc <- ave(ta$ncells, ta$cluster, FUN=function(x) x / sum(x) * 100)
# Normalise to percentage within each Phase
ta$ncells_phase_perc <- ave(ta$ncells, ta$Phase, FUN=function(x) x / sum(x) * 100)
# Relative to total cells (double normalization)
# Observed proportion of cells per Phase from each condition / expected proportion of cells per Phase from each condition if equally distributed
# =1 equal distribution to the expected. >1 more cells in the Phase from the condition than expected, <1 less cells in the Phase from the condition than expected
total_cells <- sum(ta$ncells)
ta$composition_normalised_for_library_size <- (ta$ncells / ave(ta$ncells, ta$cluster, FUN=sum)) / 
                         (ave(ta$ncells, ta$Phase, FUN=sum) / total_cells)
                        
write.csv(ta, file=paste0(tabDir, "cells_per_Phase_cond_multiple_normalizations.csv"))
g <- ggplot(aes(Phase, composition_normalised_for_library_size, fill=Phase), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~cluster, ncol=4) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

pdf(paste0(figDir, "Phase_composition_cluster_normalised_for_library_size.pdf"), width=12, height=8)
print(g)
dev.off()

In [ ]:
### Run MAGIC
library(reticulate)
use_condaenv("magic2", required = TRUE)
use_python("/data/ebaird/miniconda3/envs/magic2/bin/python", required = TRUE)
library(Rmagic)
mag <- Rmagic::magic(seu, verbose=F)
seu@assays$MAGIC_SCT <- mag@assays$MAGIC_SCT

In [ ]:
# save(seu, file=paste0(repDir, "magicRDS.RData"))
load(file=paste0(repDir, "magicRDS.RData"))

In [ ]:
# Bulk RNAseq candidates - using ALL filtered genes
bulkrnaseq <- list()

# Read RNA-seq data
tmp <- read.csv(paste0(refsDir, "/o3.GAL_NDE.wRi.vs.FLP_NDE.no.transposons.csv"), row.names = 1)
tmp$gene <- sapply(strsplit(rownames(tmp), "_"), `[`, 2)

# GALvsFLP with logFC filtering
bulkrnaseq$FLPvsGAL_T0_DW.logfc0.5.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logFC > 0.5]
# bulkrnaseq$FLPvsGAL_T0_DW.logfc1.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logFC > 1]
bulkrnaseq$FLPvsGAL_T0_UP.logfc0.5.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logFC < -0.5]
# bulkrnaseq$FLPvsGAL_T0_UP.logfc1.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logFC < -1]

# GALvsFLP with logCPM filtering
# bulkrnaseq$FLPvsGAL_T0_DW.logcpm1.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logCPM > 1 & tmp$logFC > 0]
# bulkrnaseq$FLPvsGAL_T0_DW.logcpm5.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logCPM > 5 & tmp$logFC > 0]
# bulkrnaseq$FLPvsGAL_T0_UP.logcpm1.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logCPM > 1 & tmp$logFC < 0]
# bulkrnaseq$FLPvsGAL_T0_UP.logcpm5.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logCPM > 5 & tmp$logFC < 0]

# GALvsFLP with FDR filtering
# bulkrnaseq$FLPvsGAL_T0_DW.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logFC > 0]
# bulkrnaseq$FLPvsGAL_T0_DW.FDR0.1 <- tmp$gene[tmp$FDR < 0.1 & tmp$logFC > 0]
# bulkrnaseq$FLPvsGAL_T0_UP.FDR0.05 <- tmp$gene[tmp$FDR < 0.05 & tmp$logFC < 0]
# bulkrnaseq$FLPvsGAL_T0_UP.FDR0.1 <- tmp$gene[tmp$FDR < 0.1 & tmp$logFC < 0]

# Bulk ATACseq candidates - using ALL filtered genes
bulkatacseq <- list()
# tmp_atac <- read.csv(paste0(refsDir, "/ATAC.4.Ethan.T0.GAL.vs.T0.FLP.DB.homer.anno.csv"))
tmp_atac <- data.frame(read.table(paste0(refsDir, '/ATAC.4.Ethan.T0.GAL.vs.T0.FLP.DB.homer.anno.csv'), sep='\t', header=TRUE))

# GALvsFLP with FDR filtering
# bulkatacseq$FLPvsGAL_T0_OPEN_inGAL.FDR0.1 <- tmp_atac$Nearest.PromoterID[tmp_atac$FDR < 0.1 & tmp_atac$Fold > 0 & tmp_atac$Nearest.PromoterID != '']
bulkatacseq$FLPvsGAL_T0_OPEN_inGAL.FDR0.05 <- tmp_atac$Nearest.PromoterID[tmp_atac$FDR < 0.05 & tmp_atac$Fold > 0 & tmp_atac$Nearest.PromoterID != '']
# bulkatacseq$FLPvsGAL_T0_CLOSE_inGAL.FDR0.1 <- tmp_atac$Nearest.PromoterID[tmp_atac$FDR < 0.1 & tmp_atac$Fold < 0 & tmp_atac$Nearest.PromoterID != '']
bulkatacseq$FLPvsGAL_T0_CLOSE_inGAL.FDR0.05 <- tmp_atac$Nearest.PromoterID[tmp_atac$FDR < 0.05 & tmp_atac$Fold < 0 & tmp_atac$Nearest.PromoterID != '']

# GAL vs FLP with Fold filtering
# bulkatacseq$FLPvsGAL_T0_OPEN_inGAL.Fold0.2.FDR0.05 <- tmp_atac$Nearest.PromoterID[tmp_atac$Fold > 0.2 & tmp_atac$FDR < 0.05 & tmp_atac$Nearest.PromoterID != '']
# bulkatacseq$FLPvsGAL_T0_OPEN_inGAL.Fold0.5.FDR0.05 <- tmp_atac$Nearest.PromoterID[tmp_atac$Fold > 0.5 & tmp_atac$FDR < 0.05 & tmp_atac$Nearest.PromoterID != '']
# bulkatacseq$FLPvsGAL_T0_CLOSE_inGAL.Fold0.2.FDR0.05 <- tmp_atac$Nearest.PromoterID[tmp_atac$Fold < -0.2 & tmp_atac$FDR < 0.05 & tmp_atac$Nearest.PromoterID != '']
# bulkatacseq$FLPvsGAL_T0_CLOSE_inGAL.Fold0.5.FDR0.05 <- tmp_atac$Nearest.PromoterID[tmp_atac$Fold < -0.5 & tmp_atac$FDR < 0.05 & tmp_atac$Nearest.PromoterID != '']

# Convert FlyBase IDs to gene symbols
library(AnnotationDbi)
library(org.Dm.eg.db)

bulkatacseq <- lapply(bulkatacseq, function(ids) {
  symbols <- mapIds(
    org.Dm.eg.db,
    keys = ids,
    column = "SYMBOL",
    keytype = "FLYBASE",
    multiVals = "first"
  )
  symbols[!is.na(symbols)]
})

# Combine all signatures
csl <- c(bulkatacseq, bulkrnaseq)

seu <- AddModuleScore(object = seu,features = csl, name = paste0(names(csl),"_ams"),assay='MAGIC_SCT')
colnames(seu@meta.data)[which(regexpr("_ams",colnames(seu@meta.data))>0)] <- names(csl)
seu[['MAGIC_SCT_AddModuleScore']]<-CreateAssayObject(t(seu@meta.data[,names(csl)]))

# # Save processed data
# save(bulkrnaseq, file = paste0(repDir, 'RNAseq_o3_candidates_all_genes.RData'))
# save(bulkatacseq, file = paste0(repDir, 'ATAC_candidates_all_genes_', format(Sys.Date(), "%Y%m%d"), '.RData'))

In [ ]:
scale_factor <- 300 / 100

# RNAseq plots
tmp <- names(bulkrnaseq)
DefaultAssay(seu) <- "MAGIC_SCT_AddModuleScore"
g <- lapply(tmp, function(i) {
        cat(i, "\t")
        FeaturePlot(seu, reduction='umap',features=gsub('_','-',i), pt.size=0.5, combine=TRUE, order=TRUE,split.by='condition', label=TRUE) & scale_colour_gradientn(colours =cols)
})
g[[length(g)+1]] <- DimPlot(seu, reduction='umap', pt.size=0.5, group.by="seurat_clusters",split.by='condition')
ggexport(ggarrange(plotlist = g, nrow = 3, ncol = 1), filename = paste0(figDir, "UMAP.bulkRNAseq.o3.GAL_NDE.wRi.vs.FLP_NDE.no.transposons.jpeg"), width=2000*scale_factor, height=1800*scale_factor, res=300)

# ATACseq plots
tmp <- names(bulkatacseq)
DefaultAssay(seu) <- "MAGIC_SCT_AddModuleScore"
g <- lapply(tmp, function(i) {
        cat(i, "\t")
        FeaturePlot(seu, reduction='umap',features=gsub('_','-',i), pt.size=0.5, combine=TRUE, order=TRUE,split.by='condition', label=TRUE) & scale_colour_gradientn(colours =cols)
})
g[[length(g)+1]] <- DimPlot(seu, reduction='umap', pt.size=0.5, group.by="seurat_clusters",split.by='condition')
ggexport(ggarrange(plotlist = g, nrow = 3, ncol = 1), filename = paste0(figDir, "UMAP.bulkATACseq.ATAC.4.Ethan.T0.GAL.vs.T0.FLP.DB.homer.anno.jpeg"), width=2000*scale_factor, height=1800*scale_factor, res=300)

In [ ]:
DefaultAssay(seu)<-'MAGIC_SCT'

dir.create(paste0(figDir,'boxplot.MAGIC_SCT_AddModuleScore'))
dir.create(paste0(figDir,'violin.MAGIC_SCT_AddModuleScore'))

# violin plots per cluster and per current annotation, split by sample
pdf(paste0(figDir, "violin.MAGIC_SCT_AddModuleScore.cell_types.pdf"), width=7.5, height=5)
for(i in names(bulkrnaseq)){
    g1 <- ggplot(seu@meta.data, aes_string(x='seurat_clusters', y = i)) + geom_violin() + theme_bw() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))
    print(g1)
}
dev.off()

pdf(paste0(figDir, "violin.MAGIC_SCT_AddModuleScore.cell_types.pdf"), width=7.5, height=5)
for(i in names(bulkatacseq)){
    g1 <- ggplot(seu@meta.data, aes_string(x='seurat_clusters', y = i)) + geom_violin() + theme_bw() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))
    print(g1)
}
dev.off()

#group.cols<-list(
#    2198='#756bb1',
#    2197='#bcbddc',
#    2196='#de2d26',
#    2199='#fc9272')

for(i in names(bulkrnaseq)){
    df <- data.frame(expression = as.numeric(seu@assays$MAGIC_SCT_AddModuleScore@data[gsub('_','-',i),]), condition = seu$condition,cluster=seu$seurat_clusters)
    png(paste0(figDir, "boxplot.MAGIC_SCT_AddModuleScore/",i,".png"), width=2000/1.75, height=3000/1.75)
    g1 <- ggplot(df, aes(x=condition, y = expression))+ geom_violin() +geom_jitter()  + theme_bw() + ggtitle(i)+facet_wrap(~cluster,ncol=4)+ theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))#+ scale_fill_manual(values = group.cols)
    print(g1)
    dev.off()
}

for(i in names(bulkatacseq)){
    df <- data.frame(expression = as.numeric(seu@assays$MAGIC_SCT_AddModuleScore@data[gsub('_','-',i),]), condition = seu$condition,cluster=seu$seurat_clusters)
    png(paste0(figDir, "boxplot.MAGIC_SCT_AddModuleScore/",i,".png"), width=2000/1.75, height=3000/1.75)
    g1 <- ggplot(df, aes(x=condition, y = expression))+ geom_violin() +geom_jitter()  + theme_bw() + ggtitle(i)+facet_wrap(~cluster,ncol=4)+ theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))#+ scale_fill_manual(values = group.cols)
    print(g1)
    dev.off()
}

In [ ]:
### Signatures

# Define all signature lists
signatures <- list(
  QNSC_brand = c('trbl', 'dpn', 'wor', 'klu', 'ase', 'pros', 'mira', 'Frq1',
   'ringer', 'Nckx30C', 'Fife', 'bol', 'eag', 'shakB', 'Nlg3', 'nAChRalpha6', 'mAChR-A',
    'nrv3', 'slo', 'Rdl', 'Fas2', 'nSyb', 'Optix', 'beat-Ic', 'Syt1', 'VAChT', 'elav', 'VGlut', 
    'Gad1', 'twit', 'Vmat', 'N', 'GFP'),
  type_i_NB = c('Rx', 'pros', 'CG31637', 'CG3788', 'sina', 'phyl', 'CG10283', 'CG7544', 'ase', 'D', 'oc'),
  type_ii_NB = c('pnt', 'Sobp', 'mEFTs', 'atk', 'slp1', 'run', 'TfAP-2', 'Optix', 'btd', 'Sp1'),
  neuronal = c('elav', 'pros', 'futsch', 'Fas2', 'Nrt', 'Vmat', 'Gad1', 'VAChT', 'VGlut'),
  hemocyte = c("Hml", "Pxn", "crq", "NimC1", "NimB4", "NimB5", "Col4a1", "PPO1", "PPO2", "lz",
   "peb", "PPO3", "msn", "mys", "cher", "regucalcin", "SPARC", "Npc2a", "Ppn", "Inos", "Nplp2",
    "CG14629", "CG34331", "srp", "drpr"),
  glycolysis_genes = c("Ald1", "Eno", "Gapdh1", "Gapdh2", "Hex-A", "Hex-C", "Pfk", "Pglym78", "Pgi", "Pgk"),
  apoptosis_genes = c("Dronc", "Drice", "Dcp-1", "rpr", "grim", "hid", "skl"),
  UPR_genes = c("Hsc70-3", "Xbp1", "crc", "Ire1"),
  proteotoxic_genes = c("Hsp26", "Hsp83", "DnaJ-1", "Hsp27", "Rpn1", "Atg8a"),
  oxidative_stress_genes = c("GstD1", "Sod2", "Cat", "SdhA", "ND-42"),
  hypoxia_genes = c("sima", "Hph", "Hsp70Aa", "tgo", "Cs2", "Ldh", 
                   "crp", "scyl", "Thor", "Qsox4", "chico"),
  urea_cycle_genes = c("CG10505", "P5CS", "Argk1", "CG10814")              

)

missing_genes <- lapply(signatures, function(genes) setdiff(genes, rownames(seu)))
missing_genes <- missing_genes[sapply(missing_genes, length) > 0]

if (length(missing_genes) > 0) {
  cat("The following genes are missing and will be excluded:\n")
  print(missing_genes)
} else {
  cat("All genes are present in the dataset.\n")
}

signatures <- lapply(signatures, function(genes) genes[genes %in% rownames(seu)])

# Create module scores and feature plots
for (signature_name in names(signatures)) {
  genes <- signatures[[signature_name]]
  
  if (length(genes) == 0) next
  
  df_expr <- FetchData(seu, vars = c(genes, "seurat_clusters", "genotype"))
  df_expr$cell <- rownames(df_expr)
  genes <- unique(genes) 
 
  seu <- AddModuleScore(object = seu, features = list(genes), name = signature_name)
  
  FeaturePlot(seu, features = paste0(signature_name, "1"), split.by = "genotype", order = T) & scale_colour_gradientn(colours = cols)
  ggsave(filename = file.path(figDir, paste0("feature_plot_", signature_name, ".jpeg")), width = 8, height = 4)
}

In [ ]:
# saveRDS(seu, file = paste0(repDir, "signatures.rds"))
seu <- readRDS(paste0(repDir, "signatures.rds"))

In [ ]:
# Gene ontology for DEGs between genotypes
library(clusterProfiler)
library(org.Dm.eg.db)
library(msigdbr)
library(gprofiler2)
library(tidyverse)

de_results <- read.csv(paste0(tabDir, "DE_results_gal_vs_flp.csv"), row.names = 1)
gal_specific_de <- rownames(de_results)[de_results$p_val < 0.05 & de_results$avg_log2FC > 0.5]
flp_specific_de <- rownames(de_results)[de_results$p_val < 0.05 & de_results$avg_log2FC < -0.5]
cluster_ids <- unique(seu$seurat_clusters)
# Marker genes for each cluster
read.csv(paste0(mainDir, "QC_clustering/tables/allMarkers_merged_clusters.csv"), row.names = 1) -> all_markers
# Gene sets for each cluster
cluster_0 <- all_markers$gene[all_markers$cluster == 0 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_1 <- all_markers$gene[all_markers$cluster == 1 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_2 <- all_markers$gene[all_markers$cluster == 2 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_3 <- all_markers$gene[all_markers$cluster == 3 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_4 <- all_markers$gene[all_markers$cluster == 4 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_5 <- all_markers$gene[all_markers$cluster == 5 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_6 <- all_markers$gene[all_markers$cluster == 6 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_7 <- all_markers$gene[all_markers$cluster == 7 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_8 <- all_markers$gene[all_markers$cluster == 8 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_9 <- all_markers$gene[all_markers$cluster == 9 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_10 <- all_markers$gene[all_markers$cluster == 10 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_11 <- all_markers$gene[all_markers$cluster == 11 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_12 <- all_markers$gene[all_markers$cluster == 12 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_13 <- all_markers$gene[all_markers$cluster == 13 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_14 <- all_markers$gene[all_markers$cluster == 14 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
cluster_15 <- all_markers$gene[all_markers$cluster == 15 & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]

allmarkers_full <- all_markers$gene[all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]

In [ ]:
# Function for enrichment analysis
run_drosophila_enrichment <- function(gene_list, name) {
  fb_ids <- mapIds(org.Dm.eg.db,
                   keys = gene_list,
                   column = "FLYBASE",
                   keytype = "SYMBOL")
  
  fb_ids <- na.omit(fb_ids)

  ego <- enrichGO(gene = fb_ids,
                OrgDb = org.Dm.eg.db,
                keyType = "FLYBASE",
                ont = "BP",
                pAdjustMethod = "BH",
                qvalueCutoff = 0.05)
  
  as.data.frame(ego)
  if (is.null(ego) || nrow(ego) == 0) {
    cat("No significant GO terms found for", name, "\n")
    return(NULL)
  }

  write.csv(as.data.frame(ego), 
            paste0(tabDir, "GO_BP", name, ".csv"), 
            row.names = FALSE)
  
  if (nrow(ego) > 0) {
    dotplot(ego, showCategory=15) + 
      ggtitle(paste("GO Enrichment:", name))
    ggsave(paste0(figDir, "GO_dotplot_BP_", name, ".png"), width=10, height=8)
  }
  
  entrez_ids <- mapIds(org.Dm.eg.db,
                       keys = gene_list,
                       column = "ENTREZID",
                       keytype = "SYMBOL",
                       multiVals = "first")

  # entrez_ids names as a list
  names(entrez_ids) <- entrez_ids
  entrez_ids <- na.omit(entrez_ids)

  # KEGG enrichment
  kegg <- enrichKEGG(gene = names(entrez_ids),
                     organism = "dme",
                     keyType = "ncbi-geneid",
                     pvalueCutoff = 0.05)
  
  as.data.frame(kegg)
  if (is.null(kegg) || nrow(ego) == 0) {
    cat("No significant GO terms found for", name, "\n")
    return(NULL)
  }

  if (!is.null(kegg) && nrow(kegg) > 0) {
    write.csv(as.data.frame(kegg), 
              paste0("KEGG_", name, ".csv"), 
              row.names = FALSE)
  }

  # Plot KEGG enrichment
  if (!is.null(kegg) && nrow(kegg) > 0) {
    dotplot(kegg, showCategory=15) + 
      ggtitle(paste("KEGG Enrichment:", name))
    ggsave(paste0(figDir, "KEGG_dotplot_", name, ".png"), width=10, height=8)
  }

  # MSigDB hallmark gene sets
  C <- msigdbr(species = "Homo sapiens", 
              collection = "C2", 
              subcollection = "CGP") %>%
    dplyr::select(gs_name, gs_id, human_gene = gene_symbol) #%>%
    # dplyr::filter(gs_id %in% c('M2116', 'M2121', 'M2122', 'M2115'))

  # Map to orthologues
  ortho_map <- gorth(
    query = unique(C$human_gene),
    source_organism = "hsapiens",
    target_organism = "dmelanogaster",
    mthreshold = 1,
    filter_na = TRUE
  ) %>% 
    as_tibble() %>%
    dplyr::select(human_gene = input, fb_ids = ortholog_ensg) %>%
    mutate(fb_ids = mapIds(org.Dm.eg.db, 
                              keys = fb_ids, 
                              column = "FLYBASE", 
                              keytype = "ENSEMBL")) %>%
    na.omit()

  # Create fly hallmark gene sets
  fly_C <- C %>%
    inner_join(ortho_map, by = "human_gene") %>%
    dplyr::select(term = gs_name, gene = fb_ids) %>%
    distinct()

  C <- enricher(
    gene = fb_ids,
    universe = keys(org.Dm.eg.db, keytype = "FLYBASE"),
    TERM2GENE = fly_C,
    pvalueCutoff = 0.05,
    pAdjustMethod = "BH",
    qvalueCutoff = 0.2
  )

  print(C)

  if (!is.null(C) && nrow(C) > 0) {
      dotplot(C, showCategory=15) + 
        ggtitle(paste("CGP Enrichment:", name))
      ggsave(paste0(figDir, "CGP_dotplot_", name, ".png"), width=10, height=8)
    }

}

# run_drosophila_enrichment(gal_specific_de, "gal_specific_de")
# run_drosophila_enrichment(flp_specific_de, "flp_specific_de")
run_drosophila_enrichment(cluster_0, "cluster_0")
run_drosophila_enrichment(cluster_1, "cluster_1")
run_drosophila_enrichment(cluster_2, "cluster_2")
run_drosophila_enrichment(cluster_3, "cluster_3")
run_drosophila_enrichment(cluster_4, "cluster_4")
run_drosophila_enrichment(cluster_5, "cluster_5")
run_drosophila_enrichment(cluster_6, "cluster_6")
run_drosophila_enrichment(cluster_7, "cluster_7")
run_drosophila_enrichment(cluster_8, "cluster_8")
run_drosophila_enrichment(cluster_9, "cluster_9")
run_drosophila_enrichment(cluster_10, "cluster_10")
run_drosophila_enrichment(cluster_11, "cluster_11")
run_drosophila_enrichment(cluster_12, "cluster_12")
run_drosophila_enrichment(cluster_13, "cluster_13")
run_drosophila_enrichment(cluster_14, "cluster_14")
run_drosophila_enrichment(cluster_15, "cluster_15")
run_drosophila_enrichment(allmarkers_full, "all_markers")

In [ ]:
differentiation_genes <- c("ase", "Hey", "tap", "dap", "insb", "pros", "cas", "fne", "elav")
stem_cell_genes <- c("Imp", "Syp", "Eip93F", "Myc", "SoxN", "HmgZ", "dpn", "klu", "chinmo")
temporal_genes <- c("Imp", "lin-28", "Syp", "Eip93F", "EcR", "foxo", "pros", "elav", "hb", "Kr", 
"pdm2", "cas", "svp", "D", "grh")
gs <- lapply(temporal_genes, function(i) FeaturePlot(seu, features=i, order=T, ncol=3, max.cutoff = "q99") & scale_colour_gradientn(colours = cols))

lay <- rbind(c(1,2,3),
             c(4,5,6),
             c(7,8,9),
             c(10,11,12),
             c(13,14,15)) 

jpeg_file <- paste0(figDir, "UMAP.temporal.genes.jpeg")
jpeg(filename = jpeg_file, width = 1800, height = 2200, res = 150)
gridExtra::grid.arrange(grobs = gs, layout_matrix = lay)
dev.off()

# jpeg(paste0(figDir, 'temporal_genes_dotplot.jpeg'), quality = 100, width = 1000, height = 700, res = 150)
# print(
#   DotPlot(seu, features = temporal_genes, dot.scale = 6) + 
#   RotatedAxis() +
#   theme(
#     axis.text.x = element_text(size = 8)
#   ) +
#   scale_color_gradientn(
#     colours = c("white", "forestgreen"),
#     limits = c(0, 1.5),
#     oob = scales::squish
#   ) +
#   ggtitle("Temporal Genes")
# )
# dev.off()

In [ ]:
#signatures to drosophila homologues
sigs_path <- paste0(refsDir, "verhaak.signatures.xlsx")
sigs <- read.xlsx(sigs_path, startRow = 1)

In [ ]:
head(sigs)
colnames(sigs)

In [ ]:
library(clusterProfiler)
library(org.Dm.eg.db)
library(msigdbr)
library(gprofiler2)
library(tidyverse)

# Assuming sigs is a data frame or tibble with human gene signatures columns

sigs_list <- unique(unlist(sigs))

# Step 2: Get ortholog mapping
ortho_map <- gorth(
  query = sigs_list,
  source_organism = "hsapiens",
  target_organism = "dmelanogaster",
  mthreshold = 1,
  filter_na = TRUE
) %>%
  as_tibble() %>%
  dplyr::select(human_gene = input, ensg = ortholog_ensg)

# Step 3: Map to FlyBase ID
ortho_map <- ortho_map %>%
  mutate(fly_gene = mapIds(
    org.Dm.eg.db,
    keys = ensg,
    column = "SYMBOL",
    keytype = "ENSEMBL",
    multiVals = "first"
  )) %>%
  filter(!is.na(fly_gene)) %>%
  dplyr::select(human_gene, fly_gene)

# Step 4: Pivot gene_df to long format
long_df <- sigs %>%
  mutate(row = row_number()) %>%
  pivot_longer(-row, names_to = "lineage", values_to = "human_gene")

# Step 5: Join orthologs
long_df <- long_df %>%
  left_join(ortho_map, by = "human_gene")

# Step 6: Pivot back to wide format
final_df <- long_df %>%
  pivot_wider(
    id_cols = row,
    names_from = lineage,
    values_from = c(human_gene, fly_gene),
    names_sep = "_"
  ) %>%
  select(-row)

# Identify fly gene columns
fly_cols <- grep("_fly$", colnames(final_df), value = TRUE)
final_df[] <- lapply(final_df, function(col) {
  if (is.list(col)) {
    sapply(col, function(x) paste(x, collapse = ";"))
  } else {
    col
  }
})
write.csv(final_df, paste0(tabDir, "verhaak_human_and_fly_orthologs.csv"), row.names = FALSE)

In [ ]:
signature_lists <- list()

for (col_name in names(final_df)) {
  clean_signature <- final_df[[col_name]][!is.na(final_df[[col_name]])]
  signature_lists[[col_name]] <- clean_signature
}

fly_cols <- grep("^fly_", names(final_df), value = TRUE)
fly_signatures <- lapply(final_df[fly_cols], function(x) x[!is.na(x)])
fly_signatures

In [ ]:
for (signature_name in names(fly_signatures)) {
  genes <- fly_signatures[[signature_name]]
  
  if (length(genes) == 0) next
  
  genes <- intersect(genes, rownames(seu))

  df_expr <- FetchData(seu, vars = c(genes, "seurat_clusters", "genotype"))
  df_expr$cell <- rownames(df_expr)
  genes <- unique(genes) 

  df_long <- df_expr %>%
    pivot_longer(cols = all_of(genes), names_to = "gene", values_to = "expression") %>%
    mutate(gene = factor(gene, levels = genes), cluster = as.factor(seurat_clusters))

  pdf(file.path(figDir, paste0("violin_grid_", signature_name, ".pdf")), width = 8, height = 10)
  
  for (genotype in unique(df_long$genotype)) {
    df_genotype <- df_long %>% filter(genotype == !!genotype)
    
    p <- ggplot(df_genotype, aes(x = cluster, y = expression)) +
      geom_violin(scale = "width", fill = "gray70", color = "black", size = 0.2, adjust = 1) +
      facet_wrap(~gene, ncol = 1, strip.position = "left", scales = "free_y") +
      theme_minimal(base_size = 10) +
      theme(
        axis.text.x = element_text(angle = 90, hjust = 1, size = 8),
        axis.text.y = element_text(size = 6),
        strip.text.y.left = element_text(angle = 0, hjust = 1),
        strip.placement = "outside",
        panel.spacing = unit(0.1, "lines")
      ) +
      labs(
        title = paste("Gene Expression per Cluster -", signature_name, "(", genotype, ")"),
        y = "Expression",
        x = "Cluster"
      )
    
    print(p)
  }
  dev.off()
 
  seu <- AddModuleScore(object = seu, features = list(genes), name = signature_name)
  
  FeaturePlot(seu, features = paste0(signature_name, "1"), split.by = "genotype", order = T) & scale_colour_gradientn(colours = cols)
  ggsave(filename = file.path(figDir, paste0("feature_plot_", signature_name, ".jpeg")), width = 8, height = 4)
}

In [ ]:
infiltrating_periphery = c("ATP1A2", "FGFR3", "LMO3", "NCAN", "FXYD1", "PSD2", "PRODH", "HIF3A",
                            "HRSP12", "KCNN3", "PPM1K", "KCNJ10", "ADCYAP1R1", "BMP7", "KAT2B", "CNTN1",
                            "SAMD9L", "SLC7A11", "ECHDC2", "FAM181B", "SALL2", "SASH1")

In [ ]:
library(clusterProfiler)
library(org.Dm.eg.db)
library(msigdbr)
library(gprofiler2)
library(tidyverse)

sigs <- infiltrating_periphery
sigs_list <- unique(unlist(sigs))

# Step 2: Get ortholog mapping
ortho_map <- gorth(
  query = sigs_list,
  source_organism = "hsapiens",
  target_organism = "dmelanogaster",
  mthreshold = 1,
  filter_na = TRUE
) %>%
  as_tibble() %>%
  dplyr::select(human_gene = input, ensg = ortholog_ensg)

# Step 3: Map to FlyBase ID
ortho_map <- ortho_map %>%
  mutate(fly_gene = mapIds(
    org.Dm.eg.db,
    keys = ensg,
    column = "SYMBOL",
    keytype = "ENSEMBL",
    multiVals = "first"
  )) %>%
  filter(!is.na(fly_gene)) %>%
  dplyr::select(human_gene, fly_gene)

In [ ]:
fly_gene_list <- unname(ortho_map$fly_gene)
seu <- AddModuleScore(object = seu, features = list(fly_gene_list), name = "infiltrating_periphery")
  
FeaturePlot(seu, features = paste0("infiltrating_periphery", "1"), split.by = "genotype", order = T) & scale_colour_gradientn(colours = cols)
ggsave(filename = file.path(figDir, paste0("feature_plot_infiltrating_periphery.jpeg")), width = 8, height = 4)